# 09 · Acesso à Energia Elétrica — 02_ANALISE
Amazonas (AM) · Censo 2010 · lê `saida/energia-am.geojson` (gerado pelo `01_ETL.ipynb`)

**Análises**
1. % de domicílios sem energia por município e por zona (rural/urbana)
2. Municípios com maior exclusão energética (ranking)
3. Variação rural × urbano para o estado como um todo
4. Correlação: distância à capital × % de domicílios sem energia

**Saídas** (em `saida/`): `indicadores-municipios.csv`, `indicadores-estado.csv`, `indicadores-mesorregiao.csv`, `correlacoes.csv`

> Só os atributos são usados (a geometria fica para o notebook 03), então o arquivo é lido como JSON e não é preciso reprojetar nada aqui.

In [20]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy import stats

DIR_SAIDA = Path("/content/drive/MyDrive/saida")
ARQ_ENERGIA = DIR_SAIDA / "energia-am.geojson"
COD_MANAUS = 1302603
TOP_N = 15
N_BOOT = 2000          # reamostragens para o intervalo de confiança da correlação
SEED = 42

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 0. Leitura e verificação

In [21]:
ARQ_ENERGIA = DIR_SAIDA / "energia-am.geojson"
with open(ARQ_ENERGIA, encoding="utf-8") as f:
    gj = json.load(f)
df = pd.DataFrame([ft["properties"] for ft in gj["features"]])

numericas = ["cd_mun", "dom_total", "dom_sem", "pct_sem",
             "dom_total_rural", "dom_sem_rural", "pct_sem_rural",
             "dom_total_urbana", "dom_sem_urbana", "pct_sem_urbana", "dist_manaus_km"]
faltam = [c for c in numericas + ["nm_mun", "nm_meso"] if c not in df.columns]
assert not faltam, f"Colunas ausentes em {ARQ_ENERGIA.name}: {faltam}. Rode o 01_ETL novamente."
df[numericas] = df[numericas].apply(pd.to_numeric, errors="coerce")
contagens = ["dom_total", "dom_sem", "dom_total_rural", "dom_sem_rural", "dom_total_urbana", "dom_sem_urbana"]
df[contagens] = df[contagens].round().astype(int)

assert len(df) == 62, f"Esperados 62 municípios; encontrei {len(df)}"
assert df["cd_mun"].is_unique
assert (df["dom_sem"] <= df["dom_total"]).all()
df["pct_dom_rural"] = 100 * df["dom_total_rural"] / df["dom_total"]   # % dos domicílios que são rurais (usado adiante)
print(f"{len(df)} municípios | {int(df['dom_total'].sum()):,} domicílios | {int(df['dom_sem'].sum()):,} sem energia")

62 municípios | 798,252 domicílios | 52,930 sem energia


In [22]:
import os

# Mount Google Drive if not already mounted
if not os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

# List files in the specified directory
print(f"Contents of {DIR_SAIDA}:")
if os.path.exists(DIR_SAIDA):
    for item in os.listdir(DIR_SAIDA):
        print(item)
else:
    print(f"Directory does not exist: {DIR_SAIDA}")

Contents of /content/drive/MyDrive/saida:
energia-am.geojson
municipios-am.geojson


## 1. % de domicílios sem energia por município e zona
`pct_sem_rural` / `pct_sem_urbana` ficam vazios (`NaN`) se o município não tiver domicílios com dados naquela zona; a célula abaixo informa quantos são.

In [23]:
cols_tab = ["cd_mun", "nm_mun", "nm_meso", "dom_total", "dom_sem", "pct_sem",
            "dom_total_rural", "dom_sem_rural", "pct_sem_rural",
            "dom_total_urbana", "dom_sem_urbana", "pct_sem_urbana"]
tab = df[cols_tab].sort_values("nm_mun").reset_index(drop=True)
tab.to_csv(DIR_SAIDA / "indicadores-municipios.csv", index=False, encoding="utf-8-sig")

print("Distribuição dos percentuais municipais:")
display(df[["pct_sem", "pct_sem_rural", "pct_sem_urbana"]].describe().T[["count", "mean", "50%", "min", "max"]]
        .rename(columns={"50%": "mediana"}))
print("Municípios sem dados urbanos:", int(df["pct_sem_urbana"].isna().sum()),
      "| sem dados rurais:", int(df["pct_sem_rural"].isna().sum()))
tab.head(10)

Distribuição dos percentuais municipais:


,count,mean,mediana,min,max
pct_sem,62.00,16.75,15.90,0.39,38.36
pct_sem_rural,62.00,36.92,34.34,5.56,71.50
pct_sem_urbana,62.00,1.59,1.41,0.14,4.24


Municípios sem dados urbanos: 0 | sem dados rurais: 0


,cd_mun,nm_mun,nm_meso,dom_total,dom_sem,pct_sem,dom_total_rural,dom_sem_rural,pct_sem_rural,dom_total_urbana,dom_sem_urbana,pct_sem_urbana
0,1300029,Alvarães,Centro Amazonense,2481,170,6.85,1118,133,11.90,1363,37,2.71
1,1300060,Amaturá,Sudoeste Amazonense,1647,377,22.89,703,365,51.92,944,12,1.27
2,1300086,Anamã,Centro Amazonense,1867,197,10.55,1061,193,18.19,806,4,0.50
3,1300102,Anori,Centro Amazonense,3060,279,9.12,1007,264,26.22,2053,15,0.73
4,1300144,Apuí,Sul Amazonense,4594,944,20.55,1879,910,48.43,2715,34,1.25
5,1300201,Atalaia do Norte,Sudoeste Amazonense,2865,1099,38.36,1616,1046,64.73,1249,53,4.24
6,1300300,Autazes,Centro Amazonense,6939,1310,18.88,3987,1265,31.73,2952,45,1.52
7,1300409,Barcelos,Norte Amazonense,4408,1264,28.68,2084,1237,59.36,2324,27,1.16
8,1300508,Barreirinha,Centro Amazonense,5137,1138,22.15,2875,1046,36.38,2262,92,4.07
9,1300607,Benjamin Constant,Sudoeste Amazonense,5950,882,14.82,2084,771,37.00,3866,111,2.87


## 2. Municípios com maior exclusão energética
Dois rankings, porque respondem a perguntas diferentes: o **percentual** mostra onde a exclusão é proporcionalmente maior (municípios pequenos oscilam mais); o **número absoluto** mostra onde há mais domicílios a atender.

In [24]:
cols_rank = ["nm_mun", "nm_meso", "dom_total", "dom_sem", "pct_sem", "pct_sem_rural", "pct_sem_urbana", "dist_manaus_km"]

rank_pct = df.sort_values("pct_sem", ascending=False).head(TOP_N)[cols_rank].reset_index(drop=True)
rank_pct.index += 1
rank_abs = df.sort_values("dom_sem", ascending=False).head(TOP_N)[cols_rank].reset_index(drop=True)
rank_abs.index += 1

conc = rank_pct["dom_sem"].sum() / df["dom_sem"].sum()
print(f"Top {TOP_N} por % concentram {conc:.1%} dos domicílios sem energia do estado.")
print(f"Top {TOP_N} por número absoluto concentram {rank_abs['dom_sem'].sum() / df['dom_sem'].sum():.1%}.")
print(f"Municípios nos dois rankings: {sorted(set(rank_pct['nm_mun']) & set(rank_abs['nm_mun']))}")
print("\nTop por percentual:")
display(rank_pct)
print("Top por número absoluto de domicílios sem energia:")
display(rank_abs)

Top 15 por % concentram 33.4% dos domicílios sem energia do estado.
Top 15 por número absoluto concentram 48.6%.
Municípios nos dois rankings: ['Barcelos', 'Manaquiri', 'Maués', 'Nova Olinda do Norte', 'São Gabriel da Cachoeira', 'São Paulo de Olivença']

Top por percentual:


,nm_mun,nm_meso,dom_total,dom_sem,pct_sem,pct_sem_rural,pct_sem_urbana,dist_manaus_km
1,Atalaia do Norte,Sudoeste Amazonense,2865,1099,38.36,64.73,4.24,"1,335.78"
2,Santa Isabel do Rio Negro,Norte Amazonense,2603,886,34.04,62.10,1.98,685.87
3,São Gabriel da Cachoeira,Norte Amazonense,7476,2377,31.80,64.20,4.16,968.78
4,Guajará,Sudoeste Amazonense,2639,815,30.88,67.82,3.31,"1,497.51"
5,Itamarati,Sudoeste Amazonense,1570,481,30.64,68.72,2.65,980.86
6,Barcelos,Norte Amazonense,4408,1264,28.68,59.36,1.16,479.87
7,São Paulo de Olivença,Sudoeste Amazonense,5325,1520,28.54,50.33,3.13,"1,059.49"
8,Ipixuna,Sudoeste Amazonense,3721,1027,27.60,48.80,2.02,"1,343.71"
9,Nova Olinda do Norte,Centro Amazonense,5570,1527,27.41,54.52,1.27,185.09
10,Maués,Centro Amazonense,9985,2631,26.35,52.35,1.70,300.42


Top por número absoluto de domicílios sem energia:


,nm_mun,nm_meso,dom_total,dom_sem,pct_sem,pct_sem_rural,pct_sem_urbana,dist_manaus_km
1,Parintins,Centro Amazonense,20657,2963,14.34,43.75,1.37,356.49
2,Maués,Centro Amazonense,9985,2631,26.35,52.35,1.70,300.42
3,São Gabriel da Cachoeira,Norte Amazonense,7476,2377,31.80,64.20,4.16,968.78
4,Manicoré,Sul Amazonense,9516,1802,18.94,33.72,1.16,401.73
5,Manaus,Centro Amazonense,460235,1778,0.39,17.36,0.29,0.00
6,Lábrea,Sul Amazonense,8114,1752,21.59,58.17,1.95,865.44
7,Manacapuru,Centro Amazonense,18144,1684,9.28,29.39,1.33,105.83
8,Nova Olinda do Norte,Centro Amazonense,5570,1527,27.41,54.52,1.27,185.09
9,São Paulo de Olivença,Sudoeste Amazonense,5325,1520,28.54,50.33,3.13,"1,059.49"
10,Coari,Centro Amazonense,15146,1423,9.40,27.33,1.17,478.36


## 3. Variação rural × urbano (estado como um todo)
O percentual do estado é calculado sobre as **contagens somadas** de todos os municípios, não como média dos percentuais municipais.

In [25]:
est = pd.DataFrame({
    "dom_total": [df["dom_total_rural"].sum(), df["dom_total_urbana"].sum(), df["dom_total"].sum()],
    "dom_sem":   [df["dom_sem_rural"].sum(),   df["dom_sem_urbana"].sum(),   df["dom_sem"].sum()],
}, index=["rural", "urbana", "total"])
est["pct_sem"] = 100 * est["dom_sem"] / est["dom_total"]
est.to_csv(DIR_SAIDA / "indicadores-estado.csv", encoding="utf-8-sig")
display(est)

p_r, p_u = est.loc["rural", "pct_sem"], est.loc["urbana", "pct_sem"]
sh_dom = est.loc["rural", "dom_total"] / est.loc["total", "dom_total"]
sh_sem = est.loc["rural", "dom_sem"] / est.loc["total", "dom_sem"]
print(f"Diferença rural − urbano: {p_r - p_u:.1f} p.p. | razão rural/urbano: {p_r / p_u:.1f}x")
print(f"A zona rural tem {sh_dom:.1%} dos domicílios, mas {sh_sem:.1%} dos domicílios sem energia.")

ambos = df.dropna(subset=["pct_sem_rural", "pct_sem_urbana"])
gap = ambos["pct_sem_rural"] - ambos["pct_sem_urbana"]
print(f"\nNo nível municipal ({len(ambos)} municípios com as duas zonas): rural > urbano em "
      f"{(gap > 0).sum()}; diferença mediana = {gap.median():.1f} p.p.")

# Por mesorregião (base do gráfico agrupado do notebook 03)
meso = df.groupby("nm_meso")[["dom_total_rural", "dom_sem_rural", "dom_total_urbana", "dom_sem_urbana", "dom_total", "dom_sem"]].sum()
meso["pct_sem_rural"]  = 100 * meso["dom_sem_rural"] / meso["dom_total_rural"]
meso["pct_sem_urbana"] = 100 * meso["dom_sem_urbana"] / meso["dom_total_urbana"]
meso["pct_sem"]        = 100 * meso["dom_sem"] / meso["dom_total"]
meso.to_csv(DIR_SAIDA / "indicadores-mesorregiao.csv", encoding="utf-8-sig")
meso[["dom_total", "dom_sem", "pct_sem", "pct_sem_rural", "pct_sem_urbana"]]

,dom_total,dom_sem,pct_sem
rural,140422,48488,34.53
urbana,657830,4442,0.68
total,798252,52930,6.63


Diferença rural − urbano: 33.9 p.p. | razão rural/urbano: 51.1x
A zona rural tem 17.6% dos domicílios, mas 91.6% dos domicílios sem energia.

No nível municipal (62 municípios com as duas zonas): rural > urbano em 62; diferença mediana = 32.9 p.p.


,dom_total,dom_sem,pct_sem,pct_sem_rural,pct_sem_urbana
nm_meso,,,,,
Centro Amazonense,652678,25424,3.90,27.56,0.49
Norte Amazonense,21683,5364,24.74,50.94,2.08
Sudoeste Amazonense,64230,11332,17.64,45.04,1.83
Sul Amazonense,59661,10810,18.12,40.96,1.83


## 4. Correlação: distância a Manaus × % de domicílios sem energia
- **Spearman** é a medida principal (a distribuição dos percentuais é assimétrica e há valores extremos); o Pearson aparece só para comparação.
- O intervalo de confiança de 95% do Spearman vem de *bootstrap* (com 62 municípios, o p-valor sozinho engana).
- Manaus (distância 0, capital, quase metade dos domicílios) é excluída numa versão de sensibilidade.
- A **distância é em linha reta**, não fluvial, que é o acesso real no AM.

In [26]:
rng = np.random.default_rng(SEED)

def correl(x, y, rotulo):
    m = x.notna() & y.notna()
    xs, ys = x[m].to_numpy(), y[m].to_numpy()
    n = len(xs)
    rho, p = stats.spearmanr(xs, ys)
    r, pr = stats.pearsonr(xs, ys)
    boots = []
    for _ in range(N_BOOT):
        i = rng.integers(0, n, n)
        boots.append(stats.spearmanr(xs[i], ys[i])[0])
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    return {"análise": rotulo, "n": n, "spearman": rho, "ic95_inf": lo, "ic95_sup": hi,
            "p_spearman": p, "pearson": r, "p_pearson": pr}

sem_capital = df["cd_mun"] != COD_MANAUS
d = df[sem_capital]
res = pd.DataFrame([
    correl(df["dist_manaus_km"], df["pct_sem"],           "% sem energia (total) — todos os municípios"),
    correl(d["dist_manaus_km"],  d["pct_sem"],            "% sem energia (total) — sem Manaus"),
    correl(d["dist_manaus_km"],  d["pct_sem_rural"],      "% sem energia rural — sem Manaus"),
    correl(d["dist_manaus_km"],  d["pct_sem_urbana"],     "% sem energia urbana — sem Manaus"),
])
res.to_csv(DIR_SAIDA / "correlacoes.csv", index=False, encoding="utf-8-sig")
res.round(3)

,análise,n,spearman,ic95_inf,ic95_sup,p_spearman,pearson,p_pearson
0,% sem energia (total) — todos os municípios,62,0.40,0.16,0.61,0.00,0.44,0.00
1,% sem energia (total) — sem Manaus,61,0.37,0.13,0.58,0.00,0.42,0.00
2,% sem energia rural — sem Manaus,61,0.50,0.27,0.68,0.00,0.54,0.00
3,% sem energia urbana — sem Manaus,61,0.46,0.22,0.64,0.00,0.48,0.00


### 4.1 A distância explica ou é só um retrato da ruralidade?
Municípios distantes tendem a ser mais rurais, e a zona rural é onde falta energia. Abaixo, a correlação entre a **% de domicílios rurais** e as outras duas variáveis, e a correlação parcial da distância controlando a ruralidade.

In [27]:
def partial_spearman(x, y, z):
    """Spearman parcial de x e y controlando z (por ranks e resíduos de regressão linear)."""
    m = x.notna() & y.notna() & z.notna()
    rx, ry, rz = (v[m].rank().to_numpy() for v in (x, y, z))
    def resid(a, b):
        beta = np.polyfit(b, a, 1)
        return a - np.polyval(beta, b)
    return stats.pearsonr(resid(rx, rz), resid(ry, rz))[0]

rho_dist_rur  = stats.spearmanr(d["dist_manaus_km"], d["pct_dom_rural"])[0]
rho_rur_sem   = stats.spearmanr(d["pct_dom_rural"],  d["pct_sem"])[0]
rho_parcial   = partial_spearman(d["dist_manaus_km"], d["pct_sem"], d["pct_dom_rural"])
rho_simples   = stats.spearmanr(d["dist_manaus_km"], d["pct_sem"])[0]

print(f"Distância × % domicílios rurais:              ρ = {rho_dist_rur:.2f}")
print(f"% domicílios rurais × % sem energia:           ρ = {rho_rur_sem:.2f}")
print(f"Distância × % sem energia (simples):           ρ = {rho_simples:.2f}")
print(f"Distância × % sem energia | ruralidade fixa:   ρ = {rho_parcial:.2f}")

Distância × % domicílios rurais:              ρ = -0.21
% domicílios rurais × % sem energia:           ρ = 0.41
Distância × % sem energia (simples):           ρ = 0.37
Distância × % sem energia | ruralidade fixa:   ρ = 0.52


## Leitura dos resultados — o que dizer e o que não dizer
- Correlação **não é causalidade** e aqui o nível é o município (falácia ecológica): não se pode concluir nada sobre domicílios individuais.
- Os dados são de **2010**; a Luz para Todos e a expansão de sistemas isolados mudaram o quadro. Conclusões sobre "hoje" exigem dado atual (ex.: ANEEL).
- Os setores omitidos por sigilo (veja o `01_ETL`) são poucos, mas se concentram em áreas muito pequenas ou esparsas, justamente onde a exclusão pode ser maior.
- Percentuais de municípios com poucos domicílios são instáveis: olhe sempre `dom_total` junto do percentual.